## 1. Instalasi Library
Menginstal PySpark dan XGBoost sebelum memuat semua modul yang dibutuhkan.

In [6]:
# Install PySpark dan XGBoost versi Spark
!pip install -q pyspark xgboost

## 2. Import Modul
Semua library dan modul yang dibutuhkan dimuat dalam satu tempat.

In [7]:
import os
import time

# Modul PySpark Core & SQL
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hour, dayofweek, to_timestamp

# Modul PySpark Machine Learning (Feature Engineering & Pipeline)
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

# Modul Evaluasi Model
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Modul XGBoost untuk Spark
from xgboost.spark import SparkXGBClassifier

print("[+] Semua modul berhasil di-import!")

[+] Semua modul berhasil di-import!


## 3. Persiapan & Unduh Dataset
Menggunakan Kaggle API untuk mengunduh dataset secara langsung ke penyimpanan sementara Google Colab.

In [8]:
# Konfigurasi Kredensial Kaggle API
os.environ['KAGGLE_USERNAME'] = "ezrahafizh"
os.environ['KAGGLE_KEY'] = "KGAT_b829d60a8bac84e5bc8460f18309985b"

print("[*] Mengunduh spesifik file HI-Medium_Trans.csv dari Kaggle...")
!kaggle datasets download -d ealtman2019/ibm-transactions-for-anti-money-laundering-aml -f HI-Medium_Trans.csv

print("[*] Mengekstrak file ZIP...")
!unzip -q -o HI-Medium_Trans.csv.zip

# Membersihkan file zip sisa untuk menghemat penyimpanan
!rm HI-Medium_Trans.csv.zip

print("\n[+] MANTAP! Dataset spesifik siap digunakan di: /content/HI-Medium_Trans.csv")

[*] Mengunduh spesifik file HI-Medium_Trans.csv dari Kaggle...
Dataset URL: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml
License(s): Community Data License Agreement - Sharing - Version 1.0
100% 2.82G/2.82G [00:48<00:00, 62.3MB/s]

[*] Mengekstrak file ZIP...
unzip:  cannot find or open HI-Medium_Trans.csv.zip, HI-Medium_Trans.csv.zip.zip or HI-Medium_Trans.csv.zip.ZIP.
rm: cannot remove 'HI-Medium_Trans.csv.zip': No such file or directory

[+] MANTAP! Dataset spesifik siap digunakan di: /content/HI-Medium_Trans.csv


## 4. Inisialisasi PySpark
Mengaktifkan PySpark dengan alokasi memori yang dioptimalkan untuk dataset besar.

In [9]:
# Atur environment variable untuk Java
java_path = os.popen("readlink -f /usr/bin/java | sed 's:/bin/java::'").read().strip()
os.environ["JAVA_HOME"] = java_path

print(f"Java ditemukan di: {java_path}")

# Membangun Spark Session
try:
    if 'spark' in locals():
        spark.stop()

    spark = SparkSession.builder \
        .master("local[*]") \
        .appName("IBM-AML-XGBoost") \
        .config("spark.driver.memory", "8g") \
        .config("spark.executor.memory", "8g") \
        .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
        .getOrCreate()

    print("Spark Session Aktif untuk XGBoost")
except Exception as e:
    print(f"Spark Session masih error: {e}")

Java ditemukan di: /usr/lib/jvm/java-17-openjdk-amd64
Spark Session Aktif untuk XGBoost


## 5. Memuat Data & EDA Singkat
Membaca data secara langsung dari CSV.

In [10]:
csv_path = '/content/HI-Medium_Trans.csv'

print("[*] Memuat data dari format CSV...")
df = spark.read.csv(csv_path, header=True, inferSchema=True)

print(f"[+] Total baris data: {df.count():,}")

print("\n[*] Menghitung distribusi kelas target (Is Laundering)...")
df.groupBy("Is Laundering").count().show()

print("\n[+] Menampilkan 5 baris pertama data:")
df.show(5, truncate=False)

[*] Memuat data dari format CSV...
[+] Total baris data: 31,898,238

[*] Menghitung distribusi kelas target (Is Laundering)...
+-------------+--------+
|Is Laundering|   count|
+-------------+--------+
|            1|   35230|
|            0|31863008|
+-------------+--------+


[+] Menampilkan 5 baris pertama data:
+----------------+---------+---------+-------+---------+---------------+------------------+-----------+----------------+--------------+-------------+
|Timestamp       |From Bank|Account2 |To Bank|Account4 |Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|
+----------------+---------+---------+-------+---------+---------------+------------------+-----------+----------------+--------------+-------------+
|2022/09/01 00:17|20       |800104D70|20     |800104D70|6794.63        |US Dollar         |6794.63    |US Dollar       |Reinvestment  |0            |
|2022/09/01 00:02|3196     |800107150|3196   |800107150|7739.29        |US Dollar  

## 6. Preprocessing & Feature Engineering
Melakukan transformasi waktu, mengonversi teks menjadi angka (StringIndexer), dan menggabungkannya ke dalam vektor.

In [11]:
print("Memulai tahapan Feature Engineering...")

# 1. Ekstrak Waktu dari Kolom Timestamp
df_cleaned = df.withColumn("Parsed_Time", to_timestamp(col("Timestamp"), "yyyy/MM/dd HH:mm")) \
               .withColumn("Hour", hour(col("Parsed_Time"))) \
               .withColumn("DayOfWeek", dayofweek(col("Parsed_Time"))) \
               .drop("Timestamp", "Parsed_Time")

# 2. Transformasi Kolom Teks (Kategorikal) Menjadi Angka Index
kolom_teks = ["Payment Format", "Payment Currency", "Receiving Currency", "Account2", "Account4"]
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_Indexed", handleInvalid="keep") for c in kolom_teks]

pipeline = Pipeline(stages=indexers)
print("[*] Melakukan indexing pada kolom teks (tunggu beberapa saat)...")
df_transformed = pipeline.fit(df_cleaned).transform(df_cleaned)

# Drop kolom teks asli
df_final = df_transformed.drop(*kolom_teks)

# 3. Menggabungkan fitur ke dalam vektor
print("[*] Menggabungkan kolom fitur menjadi satu vektor tunggal...")
kolom_fitur = [
    'From Bank', 'To Bank', 'Amount Received', 'Amount Paid',
    'Hour', 'DayOfWeek', 'Payment Format_Indexed',
    'Payment Currency_Indexed', 'Receiving Currency_Indexed',
    'Account2_Indexed', 'Account4_Indexed'
]

assembler = VectorAssembler(inputCols=kolom_fitur, outputCol="features")
df_ml = assembler.transform(df_final).select("features", col("Is Laundering").alias("label"))

# 4. Split Data
print("[*] Membagi data menjadi Train Set (80%) dan Test Set (20%)...")
train_data, test_data = df_ml.randomSplit([0.8, 0.2], seed=42)

print("\n[+] FEATURE ENGINEERING SELESAI!")
print(f"Jumlah Data Training: {train_data.count():,}")
print(f"Jumlah Data Testing: {test_data.count():,}")

Memulai tahapan Feature Engineering...
[*] Melakukan indexing pada kolom teks (tunggu beberapa saat)...
[*] Menggabungkan kolom fitur menjadi satu vektor tunggal...
[*] Membagi data menjadi Train Set (80%) dan Test Set (20%)...

[+] FEATURE ENGINEERING SELESAI!
Jumlah Data Training: 25,519,244
Jumlah Data Testing: 6,378,994


## 7. Downsampling
Menangani masalah kelas data yang tidak seimbang (*imbalanced data*) agar model tidak bias.

In [12]:
print("Melakukan downsampling data latih untuk model XGBoost...")

# Pisahkan kelas 1 dan kelas 0 dari data training asli
train_laundering = train_data.filter(train_data.label == 1)
train_normal = train_data.filter(train_data.label == 0)

# Ambil sampel acak dari data normal agar seimbang dengan data laundering
fraction = 100000 / train_normal.count()
train_normal_sampled = train_normal.sample(withReplacement=False, fraction=fraction, seed=42)

# Satukan data untuk training
xgb_train_data = train_laundering.union(train_normal_sampled)

print(f"Data training asli: {train_data.count():,} baris")
print(f"Data training XGBoost setelah disederhanakan: {xgb_train_data.count():,} baris")

Melakukan downsampling data latih untuk model XGBoost...
Data training asli: 25,519,244 baris
Data training XGBoost setelah disederhanakan: 128,621 baris


## 8. Training Model XGBoost
Melatih klasifikasi XGBoost secara terdistribusi.

In [13]:
print("Melatih model XGBoost terdistribusi (SparkXGBClassifier)...")
start_train = time.time()

# Inisialisasi model
xgb = SparkXGBClassifier(
    features_col="features",
    label_col="label",
    max_depth=6,
    learning_rate=0.1,
    num_workers=2,
    scale_pos_weight=1.0
)

xgb_model = xgb.fit(xgb_train_data)

print(f"Training XGBoost selesai dalam {time.time() - start_train:.2f} detik!")

# Menyimpan model (Di penyimpanan lokal colab)
model_save_path = '/content/xgboost_aml_model'
xgb_model.write().overwrite().save(model_save_path)
print(f"Model berhasil disimpan di: {model_save_path}")

Melatih model XGBoost terdistribusi (SparkXGBClassifier)...


INFO:XGBoost-PySpark:Running xgboost-3.2.0 on 2 workers with
	booster params: {'objective': 'binary:logistic', 'device': 'cpu', 'learning_rate': 0.1, 'max_depth': 6, 'scale_pos_weight': 1.0, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 100}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
INFO:XGBoost-PySpark:Finished xgboost training!


Training XGBoost selesai dalam 2156.31 detik!
Model berhasil disimpan di: /content/xgboost_aml_model


## 9. Evaluasi Model
Mengukur performa deteksi pencucian uang menggunakan perhitungan *Confusion Matrix* yang aman dari kendala *Out of Memory*.

In [15]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import time

start_eval = time.time()

print("Melakukan Prediksi pada data testing skala penuh...")
predictions = xgb_model.transform(test_data)

# 1. Menghitung ROC-AUC (1x Eksekusi)
print("\n[*] Menghitung skor ROC-AUC...")
evaluator = BinaryClassificationEvaluator(rawPredictionCol="probability", metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)
print(f"[+] HASIL AKHIR EVALUASI - XGBoost ROC-AUC: {auc:.4f}")

# 2. Menghitung Confusion Matrix dengan GROUPBY (Hanya 1x Eksekusi untuk 4 nilai)
print("\n[*] Mengekstraksi Confusion Matrix (Metode One-Pass Super Cepat)...")
cm_df = predictions.groupBy("label", "prediction").count().collect()

# Inisialisasi variabel
tp = tn = fp = fn = 0

# Ekstrak nilai dari hasil aggregasi
for row in cm_df:
    if row["label"] == 1.0 and row["prediction"] == 1.0:
        tp = row["count"]
    elif row["label"] == 0.0 and row["prediction"] == 0.0:
        tn = row["count"]
    elif row["label"] == 0.0 and row["prediction"] == 1.0:
        fp = row["count"]
    elif row["label"] == 1.0 and row["prediction"] == 0.0:
        fn = row["count"]

# 3. Kalkulasi Metrik Akhir
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
accuracy = (tp + tn) / (tn + fp + fn + tp) if (tn + fp + fn + tp) > 0 else 0

print("\n" + "="*50)
print("     Classification Report - XGBoost (PySpark)")
print("="*50)
print(f"Accuracy     : {accuracy:.4f}")
print(f"Precision    : {precision:.4f}")
print(f"Recall (TPR) : {recall:.4f}")
print(f"F1-Score     : {f1_score:.4f}")
print("="*50)
print(f"Detail:\nTrue Positive (TP)  = {tp}\nFalse Positive (FP) = {fp}\nFalse Negative (FN) = {fn}\nTrue Negative (TN)  = {tn}")
print("="*50)

print(f"\nTotal waktu evaluasi: {time.time() - start_eval:.2f} detik!")

Melakukan Prediksi pada data testing skala penuh...

[*] Menghitung skor ROC-AUC...
[+] HASIL AKHIR EVALUASI - XGBoost ROC-AUC: 0.9803

[*] Mengekstraksi Confusion Matrix (Metode One-Pass Super Cepat)...

     Classification Report - XGBoost (PySpark)
Accuracy     : 0.9666
Precision    : 0.0269
Recall (TPR) : 0.8399
F1-Score     : 0.0522
Detail:
True Positive (TP)  = 5872
False Positive (FP) = 212153
False Negative (FN) = 1119
True Negative (TN)  = 6159850

Total waktu evaluasi: 1557.91 detik!
